# Lab Week 9 — Transformers for Neural Machine Translation

We will compare three different models: 
1. RNN LSTM
2. RNN LSTM with Attention
3. Transformer with Multi-head Attention and Positional Encoding.  


**Main tasks:**
In this lab, we will build a Transformer model for English → Spanish neural machine translation using multi-head attention and positional encoding.
1. Load and prepare an English–Spanish dataset.
2. Convert text into token ID sequences.
3. Build a Transformer encoder-decoder model.
4. Train the model.
5. Test the model with sample English sentences.

## 1. Set up and Import libraries

1. Revert Keras to version 2. `os.environ["TF_USE_LEGACY_KERAS"] = "1"` and import the `tf_keras` package. This ensures that `tf.keras` points to `tf_keras`, which is Keras 2.*.
2. TensorFlow ≥ 2.8

In [1]:
# Make sure the version of Python is 3.7 or above:
import sys
assert sys.version_info >= (3, 7)

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

from pathlib import Path
import numpy as np

try:
    import tf_keras
except ImportError:
    pass


from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")
print("TensorFlow version:", tf.__version__)

2026-06-08 08:33:30.045131: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-08 08:33:30.065770: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-08 08:33:30.065813: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-08 08:33:30.080082: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-08 08:33:30.768270: W tensorflow/compiler/tf

TensorFlow version: 2.16.2


## Data Preparing

### 1. Downloads the Spanish-English dataset

In [2]:
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, cache_dir="datasets",
                               extract=True)
text = (Path(path).with_name("spa-eng") / "spa.txt").read_text()

print(text[:100])
text[:100]

Go.	Ve.
Go.	Vete.
Go.	Vaya.
Go.	Váyase.
Hi.	Hola.
Run!	¡Corre!
Run.	Corred.
Who?	¿Quién?
Fire!	¡Fueg


'Go.\tVe.\nGo.\tVete.\nGo.\tVaya.\nGo.\tVáyase.\nHi.\tHola.\nRun!\t¡Corre!\nRun.\tCorred.\nWho?\t¿Quién?\nFire!\t¡Fueg'

### 2.1 Text vectorization for source and target languages.
The English and Spanish text must be converted into integer token sequences.

Important implementation points:
- Keep `[start]` and `[end]` tokens in Spanish.
- Lowercase and remove most punctuation.
- Use fixed sequence lengths for batching.

**Use the 'pairs' to store the data shown below:**
```bash
[
    [English sentence, Translated sentence],
    [English sentence, Translated sentence],
    [English sentence, Translated sentence]
]
```

In [3]:
import numpy as np

text = text.replace("¡", "").replace("¿", "")

pairs = [line.split("\t") for line in text.splitlines()]
print(pairs[:100])

np.random.seed(42)  # extra code – ensures reproducibility on CPU
np.random.shuffle(pairs)
sentences_en, sentences_es = zip(*pairs)  # separates the pairs into 2 lists

#Check the sentences in English and Spanish to verify that they are correctly paired.
for i in range(3):
    print(sentences_en[i], "=>", sentences_es[i])

[['Go.', 'Ve.'], ['Go.', 'Vete.'], ['Go.', 'Vaya.'], ['Go.', 'Váyase.'], ['Hi.', 'Hola.'], ['Run!', 'Corre!'], ['Run.', 'Corred.'], ['Who?', 'Quién?'], ['Fire!', 'Fuego!'], ['Fire!', 'Incendio!'], ['Fire!', 'Disparad!'], ['Help!', 'Ayuda!'], ['Help!', 'Socorro! Auxilio!'], ['Help!', 'Auxilio!'], ['Jump!', 'Salta!'], ['Jump.', 'Salte.'], ['Stop!', 'Parad!'], ['Stop!', 'Para!'], ['Stop!', 'Pare!'], ['Wait!', 'Espera!'], ['Wait.', 'Esperen.'], ['Go on.', 'Continúa.'], ['Go on.', 'Continúe.'], ['Hello!', 'Hola.'], ['I ran.', 'Corrí.'], ['I ran.', 'Corría.'], ['I try.', 'Lo intento.'], ['I won!', 'He ganado!'], ['Oh no!', 'Oh, no!'], ['Relax.', 'Tomátelo con soda.'], ['Smile.', 'Sonríe.'], ['Attack!', 'Al ataque!'], ['Attack!', 'Atacad!'], ['Get up.', 'Levanta.'], ['Go now.', 'Ve ahora mismo.'], ['Got it!', 'Lo tengo!'], ['Got it?', 'Lo pillas?'], ['Got it?', 'Entendiste?'], ['He ran.', 'Él corrió.'], ['Hop in.', 'Métete adentro.'], ['Hug me.', 'Abrázame.'], ['I fell.', 'Me caí.'], ['I know

In [4]:
vocab_size = 1000
max_length = 50
embed_size = 128

text_vec_layer_en = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=max_length
)

text_vec_layer_es = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=max_length
)

text_vec_layer_en.adapt(sentences_en)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es])

print("English vocabulary size:", len(text_vec_layer_en.get_vocabulary()))
print("Spanish vocabulary size:", len(text_vec_layer_es.get_vocabulary()))
print("First 10 English tokens:", text_vec_layer_en.get_vocabulary()[:10])
print("First 10 Spanish tokens:", text_vec_layer_es.get_vocabulary()[:10])

2026-06-08 08:33:32.166304: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2a:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 08:33:32.207557: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2a:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 08:33:32.207621: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2a:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 08:33:32.209563: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2a:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 08:33:32.209622: I external/local_xla/xla/stream_executor

English vocabulary size: 1000
Spanish vocabulary size: 1000
First 10 English tokens: ['', '[UNK]', 'the', 'i', 'to', 'you', 'tom', 'a', 'is', 'he']
First 10 Spanish tokens: ['', '[UNK]', 'startofseq', 'endofseq', 'de', 'que', 'a', 'no', 'tom', 'la']


### 2.2 Prepare Training and Validation Dataset

Note: 
1. When you ouput X_train, you find "b" is begining for each sentence, which means means this is a bytes string, not a normal Python Unicode string.
And if you find some output looks like "Qu\xc3\xa9", that means TensorFlow is displaying the string internally as UTF-8 encoded bytes.
For example: "Qué" will be printed as "b'Qu\xc3\xa9"

2. The output vector represents a sentence of length 50. For example: [49 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]; where 49 corresponds to "go", 1 is an [UNK] (unknown/out-of-vocabulary) word.

In [5]:
train_size = 100_000
valid_size = 18_964

X_train = tf.constant(sentences_en[:train_size])
X_valid = tf.constant(sentences_en[train_size:train_size + valid_size])

X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:train_size]])
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[train_size:train_size + valid_size]])

Y_train = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[:train_size]])
Y_valid = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[train_size:train_size + valid_size]])

print("Encoder training input:", X_train.shape,"\nThe first three of X_train", X_train[:3])
print("Decoder training input:", X_train_dec.shape,"\nThe first three of X_train_dec", X_train_dec[:3])
print("Training target:", Y_train.shape, "\nThe first three of Y_train", Y_train[:3])

Encoder training input: (100000,) 
The first three of X_train tf.Tensor([b'How boring!' b'I love sports.' b'Would you like to swap jobs?'], shape=(3,), dtype=string)
Decoder training input: (100000,) 
The first three of X_train_dec tf.Tensor(
[b'startofseq Qu\xc3\xa9 aburrimiento!' b'startofseq Adoro el deporte.'
 b'startofseq Te gustar\xc3\xada que intercambiemos los trabajos?'], shape=(3,), dtype=string)
Training target: (100000, 50) 
The first three of Y_train tf.Tensor(
[[ 25   1   3   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0]
 [  1  10   1   3   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0]
 [ 28 170   5   1  21   1   3   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0

## LSTM Model

### Build a simple encoder-decoder LSTM translation model.

In [6]:
tf.random.set_seed(42)  # extra code – ensures reproducibility on CPU

encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

embed_size = 128

encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

In [7]:
encoder = tf.keras.layers.LSTM(512, return_state=True)
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(decoder_outputs)

**Warning**: the following cell will take a while to run (possibly a couple hours if you are not using a GPU).

In [8]:
#epochs：How many times the training data was learnt
model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], 
                       outputs=[Y_proba])

model.compile(loss="sparse_categorical_crossentropy", 
              optimizer="nadam",
              metrics=["accuracy"])
model.fit(
    (X_train, X_train_dec), 
    Y_train, epochs=10,
    validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/10


2026-06-08 08:34:07.163158: W tensorflow/core/common_runtime/type_inference.cc:339] Type inference failed. This indicates an invalid graph that escaped type checking. Error message: INVALID_ARGUMENT: expected compatible input types, but input 1:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_INT32
    }
  }
}
 is neither a subtype nor a supertype of the combined inputs preceding it:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_FLOAT
    }
  }
}

	for Tuple type infernce function 0
	while inferring type of node 'cond_36/output/_23'
2026-06-08 08:34:07.641207: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1780922048.364353  111487 service.cc:145] XLA service 0x73f27805ed00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780922048.364411  111487 s

3125/3125 [==============================] - 150s 42ms/step - loss: 2.8750 - accuracy: 0.4334 - val_loss: 2.1178 - val_accuracy: 0.5331
Epoch 2/10
3125/3125 [==============================] - 120s 38ms/step - loss: 1.8004 - accuracy: 0.5852 - val_loss: 1.6206 - val_accuracy: 0.6171
Epoch 3/10
3125/3125 [==============================] - 122s 39ms/step - loss: 1.4003 - accuracy: 0.6574 - val_loss: 1.4048 - val_accuracy: 0.6611
Epoch 4/10
3125/3125 [==============================] - 121s 39ms/step - loss: 1.1737 - accuracy: 0.7012 - val_loss: 1.3206 - val_accuracy: 0.6760
Epoch 5/10
3125/3125 [==============================] - 120s 38ms/step - loss: 1.0131 - accuracy: 0.7338 - val_loss: 1.2801 - val_accuracy: 0.6863
Epoch 6/10
3125/3125 [==============================] - 121s 39ms/step - loss: 0.8820 - accuracy: 0.7617 - val_loss: 1.2794 - val_accuracy: 0.6877
Epoch 7/10
3125/3125 [==============================] - 121s 39ms/step - loss: 0.7711 - accuracy: 0.7867 - val_loss: 1.2953 - val

### Test RNN LSTM

In [9]:
def translate(sentence_en):
    translation = ""
    for word_idx in range(max_length):
        X = np.array([sentence_en])  # encoder input 
        X_dec = np.array(["startofseq " + translation])  # decoder input
        y_proba = model.predict((X, X_dec))[0, word_idx]  # last token's probas
        predicted_word_id = np.argmax(y_proba)
        predicted_word = text_vec_layer_es.get_vocabulary()[predicted_word_id]
        if predicted_word == "endofseq":
            break
        translation += " " + predicted_word
    return translation.strip()

In [10]:
translate("I like soccer")

1/1 [==============================] - 0s 34ms/step


'me gusta el fútbol'

Nice! However, the model struggles with longer sentences:

In [11]:
translate("I like soccer and also going to the beach")

1/1 [==============================] - 0s 31ms/step


'me gusta el fútbol y el trabajo'

## Attention Mechanisms

We need to feed all the encoder's outputs to the `Attention` layer, so we must add `return_sequences=True` to the encoder:

In [12]:
tf.random.set_seed(42)  # extra code – ensures reproducibility on CPU
encoder = tf.keras.layers.LSTM(
    512,
    return_sequences=True,
    return_state=True
)
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

decoder = tf.keras.layers.LSTM(512,return_sequences=True)
decoder_outputs = decoder(decoder_embeddings,initial_state=encoder_state)

# add the `Attention` layer and the output layer:
attention_layer = tf.keras.layers.Attention()
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])

output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

**Warning**: the following cell will take a while to run (possibly a couple hours if you are not using a GPU).

In [13]:
model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs],
                       outputs=[Y_proba])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
model.fit((X_train, X_train_dec), Y_train, epochs=10,
          validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/10


W0000 00:00:1780923274.669590  111395 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "NVIDIA GeForce RTX 3060" frequency: 1837 num_cores: 28 environment { key: "architecture" value: "8.6" } environment { key: "cuda" value: "12030" } environment { key: "cudnn" value: "8906" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 2359296 shared_memory_size_per_multiprocessor: 102400 memory_size: 10180624384 bandwidth: 360048000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


3125/3125 [==============================] - ETA: 0s - loss: 2.4977 - accuracy: 0.4923

W0000 00:00:1780923400.149100  111395 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "NVIDIA GeForce RTX 3060" frequency: 1837 num_cores: 28 environment { key: "architecture" value: "8.6" } environment { key: "cuda" value: "12030" } environment { key: "cudnn" value: "8906" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 2359296 shared_memory_size_per_multiprocessor: 102400 memory_size: 10180624384 bandwidth: 360048000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


3125/3125 [==============================] - 142s 43ms/step - loss: 2.4977 - accuracy: 0.4923 - val_loss: 1.6471 - val_accuracy: 0.6281
Epoch 2/10
3125/3125 [==============================] - 125s 40ms/step - loss: 1.4699 - accuracy: 0.6586 - val_loss: 1.4087 - val_accuracy: 0.6713
Epoch 3/10
3125/3125 [==============================] - 124s 40ms/step - loss: 1.2758 - accuracy: 0.6945 - val_loss: 1.3144 - val_accuracy: 0.6888
Epoch 4/10
3125/3125 [==============================] - 125s 40ms/step - loss: 1.1549 - accuracy: 0.7168 - val_loss: 1.2774 - val_accuracy: 0.6978
Epoch 5/10
3125/3125 [==============================] - 125s 40ms/step - loss: 1.0592 - accuracy: 0.7355 - val_loss: 1.2614 - val_accuracy: 0.7012
Epoch 6/10
3125/3125 [==============================] - 126s 40ms/step - loss: 0.9783 - accuracy: 0.7510 - val_loss: 1.2732 - val_accuracy: 0.7016
Epoch 7/10
3125/3125 [==============================] - 124s 40ms/step - loss: 0.9088 - accuracy: 0.7648 - val_loss: 1.2737 - val

### Test the Model

In [14]:
translate("I like soccer and also going to the beach")

W0000 00:00:1780924532.569364  111395 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "NVIDIA GeForce RTX 3060" frequency: 1837 num_cores: 28 environment { key: "architecture" value: "8.6" } environment { key: "cuda" value: "12030" } environment { key: "cudnn" value: "8906" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 2359296 shared_memory_size_per_multiprocessor: 102400 memory_size: 10180624384 bandwidth: 360048000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


1/1 [==============================] - 0s 31ms/step


'me gusta el fútbol y también a la playa'

## Attention Is All You Need: The Transformer Architecture

The Transformer uses:
1. **Token embeddings** to represent words/subwords.
2. **Positional encoding** to represent word order.
3. **Multi-head self-attention** in the encoder.
4. **Masked multi-head self-attention** in the decoder.
5. **Cross-attention** from decoder to encoder outputs.
6. Feed-forward layers, residual connections, and layer normalization.

### Build positional encoding layer
Transformer has no recurrence.
So it does not naturally know token order.
Therefore we add position information to word embeddings.

In [15]:
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_length, embed_size, dtype=tf.float32, **kwargs):
        super().__init__(dtype=dtype, **kwargs)
        assert embed_size % 2 == 0, "embed_size must be even"
        p, i = np.meshgrid(np.arange(max_length),
                           2 * np.arange(embed_size // 2))
        pos_emb = np.empty((1, max_length, embed_size))
        pos_emb[0, :, ::2] = np.sin(p / 10_000 ** (i / embed_size)).T
        pos_emb[0, :, 1::2] = np.cos(p / 10_000 ** (i / embed_size)).T
        self.pos_encodings = tf.constant(pos_emb.astype(self.dtype))
        self.supports_masking = True

    def call(self, inputs):
        batch_max_length = tf.shape(inputs)[1]
        return inputs + self.pos_encodings[:, :batch_max_length]

### Build the Transformer model with multi-head attention

In [16]:
tf.random.set_seed(42)

num_layers = 2
num_heads = 8
dropout_rate = 0.1
ffn_units = 128

encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string, name="encoder_inputs")
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string, name="decoder_inputs")

encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

positional_encoding = PositionalEncoding(max_length, embed_size)

encoder_in = positional_encoding(encoder_embeddings)
decoder_in = positional_encoding(decoder_embeddings)

encoder_pad_mask = tf.math.not_equal(encoder_input_ids, 0)[:, tf.newaxis]
decoder_pad_mask = tf.math.not_equal(decoder_input_ids, 0)[:, tf.newaxis]

batch_max_len_dec = tf.shape(decoder_input_ids)[1]
causal_mask = tf.linalg.band_part(
    tf.ones((batch_max_len_dec, batch_max_len_dec), tf.bool),
    -1,
    0
)

Z = encoder_in

for _ in range(num_layers):
    skip = Z
    Z = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_size,
        dropout=dropout_rate
    )(Z, value=Z, attention_mask=encoder_pad_mask)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

    skip = Z
    Z = tf.keras.layers.Dense(ffn_units, activation="relu")(Z)
    Z = tf.keras.layers.Dense(embed_size)(Z)
    Z = tf.keras.layers.Dropout(dropout_rate)(Z)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

encoder_outputs = Z
Z = decoder_in

for _ in range(num_layers):
    skip = Z
    Z = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_size,
        dropout=dropout_rate
    )(Z, value=Z, attention_mask=causal_mask & decoder_pad_mask)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

    skip = Z
    Z = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_size,
        dropout=dropout_rate
    )(Z, value=encoder_outputs, attention_mask=encoder_pad_mask)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

    skip = Z
    Z = tf.keras.layers.Dense(ffn_units, activation="relu")(Z)
    Z = tf.keras.layers.Dense(embed_size)(Z)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

### Train the Model

In [17]:
Y_proba = tf.keras.layers.Dense(vocab_size, activation="softmax")(Z)

model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs],
                       outputs=[Y_proba])
model.compile(loss="sparse_categorical_crossentropy", 
              optimizer="nadam",
              metrics=["accuracy"])
model.fit((X_train, X_train_dec), Y_train, 
          epochs=10,
          validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/10
3125/3125 [==============================] - 352s 107ms/step - loss: 2.7359 - accuracy: 0.4575 - val_loss: 1.9208 - val_accuracy: 0.5802
Epoch 2/10
3125/3125 [==============================] - 326s 104ms/step - loss: 1.7750 - accuracy: 0.5993 - val_loss: 1.5688 - val_accuracy: 0.6398
Epoch 3/10
3125/3125 [==============================] - 318s 102ms/step - loss: 1.5541 - accuracy: 0.6373 - val_loss: 1.4397 - val_accuracy: 0.6638
Epoch 4/10
3125/3125 [==============================] - 311s 99ms/step - loss: 1.4451 - accuracy: 0.6563 - val_loss: 1.3738 - val_accuracy: 0.6743
Epoch 5/10
3125/3125 [==============================] - 310s 99ms/step - loss: 1.3790 - accuracy: 0.6672 - val_loss: 1.3310 - val_accuracy: 0.6813
Epoch 6/10
3125/3125 [==============================] - 312s 100ms/step - loss: 1.3260 - accuracy: 0.6769 - val_loss: 1.2854 - val_accuracy: 0.6890
Epoch 7/10
3125/3125 [==============================] - 310s 99ms/step - loss: 1.2901 - accuracy: 0.6835 - val_los

### Test the Model

In [18]:
translate("I like soccer and also going to the beach")

1/1 [==============================] - 0s 47ms/step


'me gusta el fútbol y también vamos a la playa'